In [ ]:
import os
import glob
import shutil

import cv2
from tqdm.contrib.concurrent import thread_map# or thread_map

import utils
import alif_awb

%load_ext autoreload
%autoreload 2

In [ ]:
BACKGROUND_PATH = "/home/klymy/data/cam4_revB/4.2_background/"

BASE_PATH = "/home/klymy/data/cam4_revB/yolo_dataset"
BASE_DATASET = "batch1_3_margin_15"
BASE_BG_DATASET = "batch1_3_margin_15_bg"
PROCESSED_DATASETS = ["batch1_3_margin_15_bg_wb_alif", "batch1_3_margin_15_bg_wb_alif_clahe"]

SUBSETS = ["train", "val", "holdout"]

In [ ]:
utils.cp_dataset(BASE_PATH, BASE_DATASET, BASE_BG_DATASET)

In [ ]:
bg_img_paths = utils.get_files_glob(BACKGROUND_PATH, "*.tif")
len(bg_img_paths)

In [ ]:
img_pth = bg_img_paths[0]
img_pth

In [ ]:
for img_pth in bg_img_paths:
    img = cv2.imread(img_pth)
    if img is None:
        print(f"Failed to read image: {img_pth}")
        continue

    img_name = os.path.basename(img_pth)
    img_batch = os.path.dirname(img_pth).split("/")[-1]
    img_full_name =f"{img_batch}____{img_name.replace('.tif', '.jpg')}"
    cv2.imwrite(os.path.join(BASE_PATH, BASE_BG_DATASET, "images", "train", img_full_name), img)
    
    with open(os.path.join(BASE_PATH, BASE_BG_DATASET, "labels", "train", img_full_name.replace(".jpg", ".txt")), "w") as f:
        pass

In [ ]:
def process_file(pth):
    img = cv2.imread(pth)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    res = alif_awb.white_balance_vectorized(img_rgb)
    res = cv2.cvtColor(res, cv2.COLOR_RGB2BGR)
    cv2.imwrite(pth, res)

    return 1

def process_file_clahe(pth):
    img = cv2.imread(pth)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    res = alif_awb.white_balance_vectorized(img_rgb)
    res = utils.clahe(res)
    res = cv2.cvtColor(res, cv2.COLOR_RGB2BGR)
    cv2.imwrite(pth, res)

    return 1

In [ ]:
for dataset_pth, f in zip(PROCESSED_DATASETS, [process_file, process_file_clahe]):
    utils.cp_dataset(BASE_PATH, BASE_BG_DATASET, dataset_pth)
    dataset_pth = os.path.join(BASE_PATH, dataset_pth)

    img_pths = utils.get_files_glob(os.path.join(dataset_pth, "images"), "*.jpg")
    print(len(img_pths))

    r = thread_map(f, img_pths, max_workers=8)
    count = 0
    for i in r:
        count += i

    print(count == len(img_pths))